In [43]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [44]:
train = pd.read_csv("UNSW_NB15_training-set.csv" ) 


test = pd.read_csv("UNSW_NB15_testing-set.csv" )


In [45]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [46]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [47]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [48]:
y = train.pop("attack_cat").values
X = train.values

In [38]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [49]:
from sklearn.decomposition import PCA

pca = PCA(n_components=30)
principalComponents = pca.fit_transform(X)
principalDfX = pd.DataFrame(data = principalComponents)

In [50]:
principalDfX.head()
print(Counter(y))

Counter({6: 37000, 5: 18871, 3: 11132, 4: 6062, 2: 4089, 7: 3496, 0: 677, 1: 583, 8: 378, 9: 44})


In [51]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(principalDfX, y, test_size=0.2, random_state=42)

In [52]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train, y_train)


KNeighborsClassifier()

In [53]:
y_pred = neigh.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 8153, 5: 3627, 3: 2468, 4: 888, 2: 811, 7: 327, 0: 147, 1: 40, 8: 6})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [54]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   3    5   23   61   32    0    7    0    0    0]
 [   9    0   14   58   25    0   10    1    0    0]
 [  15    7  282  303   52    0  125    2    0    0]
 [  73   14  310  699  199    6  956   17    1    0]
 [  41    9   68  308  261    5  512    7    1    0]
 [   1    1    6   35    9 3599   70    2    0    0]
 [   4    1   60  842  278   13 6198   20    2    0]
 [   1    3   47  143   24    2  227  276    0    0]
 [   0    0    1   16    7    2   45    2    2    0]
 [   0    0    0    3    1    0    3    0    0    0]]
Accuracy Score : 0.6874354770146354
Report : 
              precision    recall  f1-score   support

           0       0.02      0.02      0.02       131
           1       0.00      0.00      0.00       117
           2       0.35      0.36      0.35       786
           3       0.28      0.31      0.29      2275
           4       0.29      0.22      0.25      1212
           5       0.99      0.97      0.98      3723
           6       0.76  

In [55]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.6923555757989828
Report : 
              precision    recall  f1-score   support

           0       0.04      0.04      0.04       546
           1       0.00      0.00      0.00       466
           2       0.37      0.37      0.37      3303
           3       0.29      0.32      0.31      8857
           4       0.30      0.23      0.26      4850
           5       0.99      0.96      0.98     15148
           6       0.77      0.84      0.80     29582
           7       0.80      0.36      0.50      2773
           8       0.05      0.00      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.69     65865
   macro avg       0.36      0.31      0.33     65865
weighted avg       0.69      0.69      0.68     65865



In [ ]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train, y_train)

In [16]:
y_pred=clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7435, 5: 3604, 3: 2146, 4: 1631, 2: 979, 7: 669, 8: 3})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [17]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   30   37   60    0    1    3    0    0]
 [   0    0    7   34   72    0    0    4    0    0]
 [   0    0  430  212  103    2    5   34    0    0]
 [   0    0  371 1558  266    7    7   65    1    0]
 [   0    0   56   99  987    0    4   65    1    0]
 [   0    0    6   85   27 3593    0   12    0    0]
 [   0    0    0    0    0    0 7418    0    0    0]
 [   0    0   72  106   94    2    0  449    0    0]
 [   0    0    7    9   21    0    0   37    1    0]
 [   0    0    0    6    1    0    0    0    0    0]]
Accuracy Score : 0.876662415740572
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.55      0.49       786
           3       0.73      0.68      0.70      2275
           4       0.61      0.81      0.69      1212
           5       1.00      0.97      0.98      3723
           6       1.00   

In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8750170803917103
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.44      0.50      0.47      3303
           3       0.71      0.68      0.69      8857
           4       0.61      0.82      0.70      4850
           5       1.00      0.96      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.65      0.65      0.65      2773
           8       0.44      0.01      0.03       303
           9       0.00      0.00      0.00        37

    accuracy                           0.88     65865
   macro avg       0.48      0.46      0.45     65865
weighted avg       0.87      0.88      0.87     65865



In [57]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train, y_train)

In [58]:
y_pred = clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({5: 8576, 6: 7789, 4: 102})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [59]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Accuracy Score : 0.5199489888868647
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.00      0.00      0.00      2275
           4       0.26      0.02      0.04      1212
           5       0.42      0.98      0.59      3723
           6       0.63      0.66      0.64      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.52     16467
   macro avg       0.13      0.17      0.13     16467
weighted avg       0.40      0.52      0.43     16467



In [60]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.5216427541182722
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.00      0.00      0.00      8857
           4       0.22      0.02      0.04      4850
           5       0.43      0.97      0.59     15148
           6       0.63      0.66      0.65     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.52     65865
   macro avg       0.13      0.17      0.13     65865
weighted avg       0.40      0.52      0.43     65865



In [61]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train, y_train)
clf

MLPClassifier(alpha=1)

In [24]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   25   23   30   45    5    3    0    0]
 [   0    0    4   24   41   45    0    3    0    0]
 [   1    0  384  250   61   56    5   29    0    0]
 [   3    0  348 1488  229  125   19   63    0    0]
 [   0    0   38  187  816  122    2   47    0    0]
 [   0    0    3   82   29 3597    1   11    0    0]
 [   0    0    0    5    1    0 7412    0    0    0]
 [   0    0   72  127  129    7    0  388    0    0]
 [   0    0    3    9   23    1    0   39    0    0]
 [   0    0    0    5    2    0    0    0    0    0]]
Accuracy Score : 0.8553470577518674
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.49      0.46       786
           3       0.68      0.65      0.67      2275
           4       0.60      0.67      0.63      1212
           5       0.90      0.97      0.93      3723
           6       1.00  

In [62]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.523586123130646
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.15      0.06      0.09      3303
           3       0.21      0.28      0.24      8857
           4       0.16      0.07      0.10      4850
           5       0.63      0.69      0.66     15148
           6       0.64      0.71      0.67     29582
           7       0.06      0.02      0.03      2773
           8       0.02      0.01      0.02       303
           9       0.00      0.00      0.00        37

    accuracy                           0.52     65865
   macro avg       0.19      0.18      0.18     65865
weighted avg       0.48      0.52      0.50     65865



In [63]:
#RandomForest
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train, y_train)
arr = forest.predict(X_train).astype(int)
print(classification_report(y_train, arr))

              precision    recall  f1-score   support

           0       0.90      0.78      0.84       546
           1       0.60      0.83      0.69       466
           2       0.41      0.89      0.57      3303
           3       0.70      0.53      0.61      8857
           4       0.76      0.62      0.68      4850
           5       0.80      0.94      0.86     15148
           6       0.94      0.89      0.92     29582
           7       0.82      0.35      0.49      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.81     65865
   macro avg       0.59      0.58      0.57     65865
weighted avg       0.83      0.81      0.80     65865



In [64]:
arr = forest.predict(X_test).astype(int)
print(classification_report(y_test, arr))

              precision    recall  f1-score   support

           0       0.15      0.05      0.07       131
           1       0.16      0.32      0.21       117
           2       0.27      0.59      0.37       786
           3       0.59      0.42      0.49      2275
           4       0.55      0.59      0.57      1212
           5       0.75      0.93      0.83      3723
           6       0.95      0.87      0.91      7418
           7       0.93      0.27      0.42       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.75     16467
   macro avg       0.44      0.40      0.39     16467
weighted avg       0.78      0.75      0.75     16467



In [65]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred  =  classifier.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.04      0.00      0.01      3303
           3       0.52      0.03      0.05      8857
           4       0.29      0.11      0.16      4850
           5       0.48      0.97      0.65     15148
           6       0.89      0.29      0.44     29582
           7       0.06      0.47      0.10      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.39     65865
   macro avg       0.23      0.19      0.14     65865
weighted avg       0.61      0.39      0.37     65865



In [66]:
y_pred  =  classifier.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.06      0.01      0.01       786
           3       0.65      0.04      0.07      2275
           4       0.31      0.10      0.15      1212
           5       0.48      0.97      0.64      3723
           6       0.89      0.29      0.44      7418
           7       0.06      0.45      0.10       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.38     16467
   macro avg       0.24      0.19      0.14     16467
weighted avg       0.63      0.38      0.37     16467



In [67]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train, y_train)
y_pred = clf_entropy.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.39      0.50      0.43      3303
           3       0.58      0.35      0.43      8857
           4       0.00      0.00      0.00      4850
           5       0.90      0.87      0.89     15148
           6       0.68      0.96      0.79     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.70     65865
   macro avg       0.25      0.27      0.25     65865
weighted avg       0.61      0.70      0.64     65865



In [68]:
y_pred = clf_entropy.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.39      0.52      0.44       786
           3       0.59      0.33      0.42      2275
           4       0.00      0.00      0.00      1212
           5       0.90      0.87      0.88      3723
           6       0.67      0.96      0.79      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.70     16467
   macro avg       0.26      0.27      0.25     16467
weighted avg       0.61      0.70      0.64     16467

